In [1]:
from pathlib import Path
import pandas as pd

In [14]:
from pathlib import Path
import pandas as pd

def combine_parquet_parts(
    folder_path: str | Path,
    base_name: str = "dataset",
    pattern: str = "*.parquet",
    output_compression: str = "gzip",
):
    """
    Combine multiple parquet part files into a single parquet file,
    ignoring files that contain 'part_all'.
    """
    folder_path = Path(folder_path)

    all_files = sorted(folder_path.glob(pattern))

    if not all_files:
        raise ValueError(f"No files found in {folder_path} with pattern {pattern}")

    print(f"[combine_parquet_parts] Found {len(all_files)} total files")

    files = []
    for f in all_files:
        if "part_all" in f.name:
            print(f"  - Skipping: {f.name} (contains 'part_all')")
        else:
            files.append(f)

    if not files:
        raise ValueError("No valid files to combine after filtering")

    print(f"[combine_parquet_parts] Combining {len(files)} files:")
    print(f"  {[f.name for f in files]}")

    # Read + concat
    df = pd.concat(
        (pd.read_parquet(f) for f in files),
        ignore_index=True
    )

    print(f"[combine_parquet_parts] Final shape: {df.shape}")

    # Save
    output_path = folder_path / f"{base_name}_part_all.parquet"
    df.to_parquet(output_path, compression=output_compression)

    print(f"[combine_parquet_parts] Saved to: {output_path}")

In [15]:
combine_parquet_parts(
    folder_path="./Armenia",
    base_name="Armenia"
)

[combine_parquet_parts] Found 4 total files
  - Skipping: Armenia_part_all.parquet (contains 'part_all')
[combine_parquet_parts] Combining 3 files:
  ['Armenia_part_1.gzip.parquet', 'Armenia_part_2.gzip.parquet', 'Armenia_part_3.gzip.parquet']
[combine_parquet_parts] Final shape: (142078, 19)
[combine_parquet_parts] Saved to: Armenia/Armenia_part_all.parquet
